In [1]:
import os
import sys

if os.getcwd().split('/')[-1] == 'genjoin':
    os.chdir('../../')
os.getcwd()

'/home/suli/Downloads/GraphQueryML'

In [2]:
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Join Order Benchmark LIGHT (JOB-LIGHT)
Data splitting improvement upon ideas outlined in `Is Your Learned Query Optimizer Behaving As You Expect? A Machine Learning Perspective` (https://arxiv.org/abs/2309.01551)

In [17]:
def query_file_to_ident(file_name):
    ident = file_name.split('.sql')[0]
    # return f"{ident[:-1].zfill(2)}{ident[-1]}"
    return ident

query_files = []
blacklisted = ['2', '12', '15', '21', '24', '27']

for file_name in os.listdir('data/queries/JOB/JOB-LIGHT'):
    if file_name.endswith('.sql'):
        query_ident = query_file_to_ident(file_name)
        if query_ident[:-1] not in blacklisted:
            query_files.append({
                "file_path": os.path.join('data/queries/JOB/JOB-LIGHT', file_name),
                "file_name": file_name,
                "query_ident": query_ident,
                "base_query": query_ident[:-1],
                "variant": query_ident[-1],
            })

df = pd.DataFrame(query_files).sort_values('query_ident').reset_index(drop=True)
df

,file_path,file_name,query_ident,base_query,variant
0,data/queries/JOB/JOB-LIGHT/10a.sql,10a.sql,10a,10,a
1,data/queries/JOB/JOB-LIGHT/10b.sql,10b.sql,10b,10,b
2,data/queries/JOB/JOB-LIGHT/10c.sql,10c.sql,10c,10,c
3,data/queries/JOB/JOB-LIGHT/11a.sql,11a.sql,11a,11,a
4,data/queries/JOB/JOB-LIGHT/11b.sql,11b.sql,11b,11,b
...,...,...,...,...,...
57,data/queries/JOB/JOB-LIGHT/9a.sql,9a.sql,9a,9,a
58,data/queries/JOB/JOB-LIGHT/9b.sql,9b.sql,9b,9,b
59,data/queries/JOB/JOB-LIGHT/9c.sql,9c.sql,9c,9,c
60,data/queries/JOB/JOB-LIGHT/9d.sql,9d.sql,9d,9,d


In [18]:
df.query_ident.values

array(['10a', '10b', '10c', '11a', '11b', '11c', '13a', '13b', '13c',
       '14a', '14b', '14c', '16a', '16b', '16c', '17a', '17b', '17c',
       '18a', '18b', '19a', '19b', '19c', '1a', '1b', '1c', '1d', '20a',
       '20b', '22a', '22b', '23a', '23b', '25a', '25b', '26a', '26b',
       '3a', '3b', '3c', '4a', '4b', '4c', '5a', '5b', '5c', '6a', '6b',
       '6c', '6d', '6e', '7a', '7b', '7c', '8a', '8b', '8c', '9a', '9b',
       '9c', '9d', '9e'], dtype=object)

In [19]:
NEW_TEST_QUERIES = {}
K = 3

### Random Split

Follow the idea of cross-validation, with `k=3`: Generate 3 folds randomly from all queries with 1/k queries in the test set, where the folds together form a partition of all queries and roughly have equal size.

In [20]:
df_shuffle = df.sample(frac=1, random_state=42)
fold_size = len(df) // K

df_random_fold_A = df_shuffle.iloc[:fold_size]
df_random_fold_B = df_shuffle.iloc[fold_size:2*fold_size]
df_random_fold_C = df_shuffle.iloc[2*fold_size:]

NEW_TEST_QUERIES['random_split_A'] = sorted(df_random_fold_A['file_name'].to_list(), key=query_file_to_ident)
NEW_TEST_QUERIES['random_split_B'] = sorted(df_random_fold_B['file_name'].to_list(), key=query_file_to_ident)
NEW_TEST_QUERIES['random_split_C'] = sorted(df_random_fold_C['file_name'].to_list(), key=query_file_to_ident)

### Base Query Split

Split all queries from the same base query "template" into either train or test, with `k` folds based on the base queries. Attempt to make the folds roughly equal in terms of the number of queries (ML perspective: amount of training/test data).

In [21]:
tmp = []

for base_query in df['base_query'].unique():
    tmp.append((base_query, df[df['base_query'] == base_query].shape[0]))

np.random.seed(42)
np.random.shuffle(tmp)

expected_size = len(df) // K
bqs_folds = [[]]

counter = 0
current_fold = 0
for l in tmp:
    counter += l[1]
    if counter <= expected_size:
        bqs_folds[current_fold].append(l)
    else:
        # check the difference of including the current base query or not
        diff_with = np.abs(counter - expected_size)
        diff_without = np.abs((counter - l[1]) - expected_size)

        # if it's closer to the expected when we include it
        if diff_with < diff_without:
            bqs_folds[current_fold].append(l)

        else:
            if current_fold < K-1:
                current_fold += 1
                counter = l[1]
                bqs_folds.append([])
            bqs_folds[current_fold].append(l)

NEW_TEST_QUERIES['base_query_split_A'] = sorted(df[df['base_query'].isin([x[0] for x in bqs_folds[0]])]['file_name'].to_list(), key=query_file_to_ident)
NEW_TEST_QUERIES['base_query_split_B'] = sorted(df[df['base_query'].isin([x[0] for x in bqs_folds[1]])]['file_name'].to_list(), key=query_file_to_ident)
NEW_TEST_QUERIES['base_query_split_C'] = sorted(df[df['base_query'].isin([x[0] for x in bqs_folds[2]])]['file_name'].to_list(), key=query_file_to_ident)

### Leave One Out Split

Split all queries into folds where one variant of each base query "template" is always in the train or test sets. In the case of `>K` variants, multiple can appear. In the case of `<K` variants, they can appear in the test set in multiple folds, e.g. 1a, 1b for 3 folds can the three test sets be [1a], [1b] and [1a], respectively

In [22]:
np.random.seed(42)

NEW_TEST_QUERIES['leave_one_out_split_A'] = []
NEW_TEST_QUERIES['leave_one_out_split_B'] = []
NEW_TEST_QUERIES['leave_one_out_split_C'] = []

for base_query, df_group in df.groupby('base_query'):
    fold_A, fold_B, fold_C = np.array_split(df_group.sample(frac=1), K)

    # ensure that even variants with just 2 queries are in every fold
    if fold_C.shape[0] == 0:
        fold_C = df_group.sample(1)

    # assign largest fold (first one in array_split) to the smallest split
    smallest_fold = np.argmin([len(NEW_TEST_QUERIES['leave_one_out_split_A']), len(NEW_TEST_QUERIES['leave_one_out_split_B']), len(NEW_TEST_QUERIES['leave_one_out_split_C'])])
    NEW_TEST_QUERIES[f'leave_one_out_split_{chr(65+smallest_fold%K)}'].extend(fold_A['file_name'].tolist())
    NEW_TEST_QUERIES[f'leave_one_out_split_{chr(65+(smallest_fold+1)%K)}'].extend(fold_B['file_name'].tolist())
    NEW_TEST_QUERIES[f'leave_one_out_split_{chr(65+(smallest_fold+2)%K)}'].extend(fold_C['file_name'].tolist())

# Sort alphabetically
NEW_TEST_QUERIES['leave_one_out_split_A'] = sorted(NEW_TEST_QUERIES['leave_one_out_split_A'], key=query_file_to_ident)
NEW_TEST_QUERIES['leave_one_out_split_B'] = sorted(NEW_TEST_QUERIES['leave_one_out_split_B'], key=query_file_to_ident)
NEW_TEST_QUERIES['leave_one_out_split_C'] = sorted(NEW_TEST_QUERIES['leave_one_out_split_C'], key=query_file_to_ident)

#### Output
Print the output in JSON format for further use in other places

In [23]:
print('NEW_TEST_QUERIES = {')
for key in NEW_TEST_QUERIES.keys():
    print(f'  "{key}": [')
    
    output_str = '    '
    prev_line_num = 0
    for query in NEW_TEST_QUERIES[key]:
        line_num, _ = np.divmod(int(query.split('.sql')[0][:-1]), 10)
        
        if line_num == prev_line_num:
            output_str += f'"{query}", '

        else:
            print(output_str)
            prev_line_num = line_num
            output_str = '    '
            output_str += f'"{query}", '

        if query == NEW_TEST_QUERIES[key][-1]:
            print(output_str)

    print(f'  ],')
print('}')

NEW_TEST_QUERIES = {
  "random_split_A": [
    
    "10a.sql", "11a.sql", "11c.sql", "13c.sql", "14a.sql", "16a.sql", "16b.sql", "17b.sql", "17c.sql", 
    "1c.sql", 
    "22b.sql", "25a.sql", "25b.sql", 
    "6b.sql", "6c.sql", "6e.sql", "8a.sql", "8c.sql", "9a.sql", "9c.sql", 
  ],
  "random_split_B": [
    
    "11b.sql", "13a.sql", "14c.sql", "17a.sql", "18b.sql", 
    "1b.sql", "1d.sql", 
    "20a.sql", "23a.sql", "23b.sql", "26b.sql", 
    "4a.sql", "4b.sql", "5a.sql", "5b.sql", "5c.sql", "7b.sql", "7c.sql", "8b.sql", "9b.sql", 
  ],
  "random_split_C": [
    
    "10b.sql", "10c.sql", "13b.sql", "14b.sql", "16c.sql", "18a.sql", "19a.sql", "19b.sql", "19c.sql", 
    "1a.sql", 
    "20b.sql", "22a.sql", "26a.sql", 
    "3a.sql", "3b.sql", "3c.sql", "4c.sql", "6a.sql", "6d.sql", "7a.sql", "9d.sql", "9e.sql", 
  ],
  "base_query_split_A": [
    
    "10a.sql", "10b.sql", "10c.sql", "11a.sql", "11b.sql", "11c.sql", "17a.sql", "17b.sql", "17c.sql", 
    "1a.sql", "1b.sql", "1c.sql", "

In [10]:
# NEW_TEST_QUERIES = {
#   "random_split_A": [
#     "1a.sql", "2a.sql", "3b.sql", "3c.sql", "4a.sql", "4b.sql", "5b.sql", "6b.sql", "6f.sql", "8a.sql", "9a.sql", "9b.sql", "9d.sql", 
#     "10c.sql", "11d.sql", "13a.sql", "13b.sql", "13d.sql", "15c.sql", "17d.sql", "18b.sql", "18c.sql", "19b.sql", "19c.sql", 
#     "20a.sql", "22a.sql", "22c.sql", "22d.sql", "23b.sql", "26b.sql", "28c.sql", 
#     "30b.sql", "30c.sql", "31c.sql", "32a.sql", "33a.sql", "33b.sql", 
#   ],
#   "random_split_B": [
#     "1d.sql", "2b.sql", "2c.sql", "2d.sql", "3a.sql", "4c.sql", "5c.sql", "6a.sql", "6c.sql", "7b.sql", "7c.sql", "8b.sql", "8c.sql", 
#     "10a.sql", "10b.sql", "11b.sql", "11c.sql", "12b.sql", "12c.sql", "14b.sql", "14c.sql", "15d.sql", "16a.sql", "16b.sql", "17f.sql", "18a.sql", "19a.sql", 
#     "20b.sql", "22b.sql", "23c.sql", "24a.sql", "25c.sql", "26a.sql", "27a.sql", "27c.sql", "28b.sql", "29b.sql", 
#   ],
#   "random_split_C": [
#     "1b.sql", "1c.sql", "5a.sql", "6d.sql", "6e.sql", "7a.sql", "8d.sql", "9c.sql", 
#     "11a.sql", "12a.sql", "13c.sql", "14a.sql", "15a.sql", "15b.sql", "16c.sql", "16d.sql", "17a.sql", "17b.sql", "17c.sql", "17e.sql", "19d.sql", 
#     "20c.sql", "21a.sql", "21b.sql", "21c.sql", "23a.sql", "24b.sql", "25a.sql", "25b.sql", "26c.sql", "27b.sql", "28a.sql", "29a.sql", "29c.sql", 
#     "30a.sql", "31a.sql", "31b.sql", "32b.sql", "33c.sql", 
#   ],
#   "base_query_split_A": [
#     "1a.sql", "1b.sql", "1c.sql", "1d.sql", "5a.sql", "5b.sql", "5c.sql", "9a.sql", "9b.sql", "9c.sql", "9d.sql", 
#     "10a.sql", "10b.sql", "10c.sql", "13a.sql", "13b.sql", "13c.sql", "13d.sql", "16a.sql", "16b.sql", "16c.sql", "16d.sql", "18a.sql", "18b.sql", "18c.sql", 
#     "20a.sql", "20b.sql", "20c.sql", "22a.sql", "22b.sql", "22c.sql", "22d.sql", "27a.sql", "27b.sql", "27c.sql", 
#     "32a.sql", "32b.sql", 
#   ],
#   "base_query_split_B": [
#     "2a.sql", "2b.sql", "2c.sql", "2d.sql", "3a.sql", "3b.sql", "3c.sql", "4a.sql", "4b.sql", "4c.sql", "6a.sql", "6b.sql", "6c.sql", "6d.sql", "6e.sql", "6f.sql", 
#     "12a.sql", "12b.sql", "12c.sql", "14a.sql", "14b.sql", "14c.sql", "17a.sql", "17b.sql", "17c.sql", "17d.sql", "17e.sql", "17f.sql", 
#     "25a.sql", "25b.sql", "25c.sql", "28a.sql", "28b.sql", "28c.sql", 
#     "30a.sql", "30b.sql", "30c.sql", 
#   ],
#   "base_query_split_C": [
#     "7a.sql", "7b.sql", "7c.sql", "8a.sql", "8b.sql", "8c.sql", "8d.sql", 
#     "11a.sql", "11b.sql", "11c.sql", "11d.sql", "15a.sql", "15b.sql", "15c.sql", "15d.sql", "19a.sql", "19b.sql", "19c.sql", "19d.sql", 
#     "21a.sql", "21b.sql", "21c.sql", "23a.sql", "23b.sql", "23c.sql", "24a.sql", "24b.sql", "26a.sql", "26b.sql", "26c.sql", "29a.sql", "29b.sql", "29c.sql", 
#     "31a.sql", "31b.sql", "31c.sql", "33a.sql", "33b.sql", "33c.sql", 
#   ],
#   "leave_one_out_split_A": [
#     "1b.sql", "1d.sql", "2c.sql", "3b.sql", "4a.sql", "5a.sql", "6c.sql", "6f.sql", "7b.sql", "8b.sql", "9c.sql", "9d.sql", 
#     "10b.sql", "11a.sql", "12a.sql", "13b.sql", "14a.sql", "15a.sql", "15d.sql", "16c.sql", "17e.sql", "17f.sql", "18c.sql", "19b.sql", 
#     "20c.sql", "21a.sql", "22c.sql", "22d.sql", "23a.sql", "24a.sql", "25c.sql", "26c.sql", "27a.sql", "28c.sql", "29b.sql", 
#     "30b.sql", "31b.sql", "32a.sql", "33b.sql", 
#   ],
#   "leave_one_out_split_B": [
#     "1a.sql", "2b.sql", "2d.sql", "3c.sql", "4c.sql", "5c.sql", "6d.sql", "6e.sql", "7a.sql", "8d.sql", "9a.sql", 
#     "10a.sql", "11b.sql", "11c.sql", "12c.sql", "13c.sql", "14b.sql", "15c.sql", "16a.sql", "16b.sql", "17a.sql", "17c.sql", "18a.sql", "19c.sql", 
#     "20b.sql", "21c.sql", "22a.sql", "23c.sql", "24a.sql", "25a.sql", "26a.sql", "27c.sql", "28b.sql", "29c.sql", 
#     "30a.sql", "31a.sql", "32a.sql", "33a.sql", 
#   ],
#   "leave_one_out_split_C": [
#     "1c.sql", "2a.sql", "3a.sql", "4b.sql", "5b.sql", "6a.sql", "6b.sql", "7c.sql", "8a.sql", "8c.sql", "9b.sql", 
#     "10c.sql", "11d.sql", "12b.sql", "13a.sql", "13d.sql", "14c.sql", "15b.sql", "16d.sql", "17b.sql", "17d.sql", "18b.sql", "19a.sql", "19d.sql", 
#     "20a.sql", "21b.sql", "22b.sql", "23b.sql", "24b.sql", "25b.sql", "26b.sql", "27b.sql", "28a.sql", "29a.sql", 
#     "30c.sql", "31c.sql", "32b.sql", "33c.sql", 
#   ],
# }

In [24]:
for k in NEW_TEST_QUERIES.keys():
    print(len(NEW_TEST_QUERIES[k]), round(len(NEW_TEST_QUERIES[k]) / len(df), 4), k)

20 0.3226 random_split_A
20 0.3226 random_split_B
22 0.3548 random_split_C
21 0.3387 base_query_split_A
18 0.2903 base_query_split_B
23 0.371 base_query_split_C
23 0.371 leave_one_out_split_A
23 0.371 leave_one_out_split_B
22 0.3548 leave_one_out_split_C


### Statistics of previous "old" LOO/BQS/RS Splits

In [13]:
TEST_QUERIES = {
    'random_split_1': ['01c', '02c', '04b', '04c', '05c', '06a', '06c', '06e', '08b', '08c', '09c', '11d', '15a', '17b', '17e', '18b', '20a', '21a', '25c', '28b', '32b', '33a'],
    'random_split_2': ['01a', '04c', '05c', '06c', '06d', '07b', '08c', '10a', '11a', '11d', '13c', '13d', '15d', '16a', '17b', '19a', '20a', '22b', '25b', '29b', '31a', '32b'],
    'random_split_3': ['02a', '03b', '06d', '09b', '10b', '11b', '11c', '13c', '13d', '16b', '18c', '19c', '21c', '22a', '22d', '26a', '26b', '27c', '28a', '28c', '30a', '33c'],
    'base_query_split_1': ['02a', '02b', '02c', '02d', '07a', '07b', '07c', '15a', '15b', '15c', '15d', '24a', '24b', '25a', '25b', '25c', '31a', '31b', '31c'],
    'base_query_split_2': ['13a', '13b', '13c', '13d', '15a', '15b', '15c', '15d', '20a', '20b', '20c', '26a', '26b', '26c', '29a', '29b', '29c', '30a', '30b', '30c', '33a', '33b', '33c'],
    'base_query_split_3': ['01a', '01b', '01c', '01d', '05a', '05b', '05c', '12a', '12b', '12c', '17a', '17b', '17c', '17d', '17e', '17f', '22a', '22b', '22c', '22d', '27a', '27b', '27c', '28a', '28b', '28c'],
    'leave_one_out_split_1': ['01c', '02a', '03b', '04a', '05a', '06b', '07c', '08c', '09c', '10b', '11b', '12c', '13b', '14a', '15b', '16c', '17c', '18b', '19a', '20c', '21c', '22b', '23b', '24a', '25a', '26c', '27c', '28a', '29b', '30a', '31b', '32b', '33c'],
    'leave_one_out_split_2': ['01d', '02d', '03a', '04b', '05c', '06d', '07a', '08c', '09c', '10a', '11a', '12a', '13d', '14b', '15b', '16a', '17f', '18a', '19d', '20a', '21b', '22c', '23b', '24b', '25a', '26a', '27b', '28c', '29a', '30b', '31a', '32b', '33b'],
    'leave_one_out_split_3': ['01c', '02d', '03b', '04a', '05c', '06d', '07b', '08a', '09a', '10c', '11d', '12a', '13a', '14b', '15a', '16d', '17b', '18b', '19d', '20b', '21a', '22a', '23b', '24a', '25b', '26a', '27a', '28b', '29c', '30a', '31a', '32a', '33c'],
}

In [14]:
for k in TEST_QUERIES.keys():
    print(len(TEST_QUERIES[k]), round(len(TEST_QUERIES[k]) / 113, 4), k)

22 0.1947 random_split_1
22 0.1947 random_split_2
22 0.1947 random_split_3
19 0.1681 base_query_split_1
23 0.2035 base_query_split_2
26 0.2301 base_query_split_3
33 0.292 leave_one_out_split_1
33 0.292 leave_one_out_split_2
33 0.292 leave_one_out_split_3
